# Supervised Fine-Tuning (SFT) with PEFT of Amazon Nova 2.0 Lite using Amazon SageMaker Training Job

## Lab 2 - Fine-tuning

In this notebook, we are going to run a Supervised Fine-Tuning job on SageMaker AI using a PyTorch Estimator with Nova recipes.

Unlike Nova 1.0 which uses the Nova Customization SDK, Nova 2.0 customization uses SageMaker Training Jobs directly with recipe-based configuration. A recipe is a YAML configuration file that provides details to SageMaker AI on how to run your model customization job.

***

## Prerequisites

If you are going to use Sagemaker in a local environment. You need access to an IAM Role with the required permissions for Sagemaker. You can find [here](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-roles.html) more about it.

In [ ]:
import sagemaker
import boto3

sess = sagemaker.Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/datasets/nova-2-lite-sft-peft"
    output_path_prefix = f"s3://{bucket_name}/{default_prefix}"
else:
    input_path = f"datasets/nova-2-lite-sft-peft"
    output_path_prefix = f"s3://{bucket_name}"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"

***

## Model fine-tuning

We now use the Nova Customization SDK to run a Supervised fine-tuning (SFT) workload with LoRA on the formatted tool-calling dataset for Amazon Nova Micro model

In [ ]:
instance_type = "ml.p5.48xlarge"
instance_count = 4

instance_type

Let's configure the runtime infrastructure using the Nova Customization SDK

In [ ]:
from amzn_nova_customization_sdk.manager.runtime_manager import SMTJRuntimeManager

runtime = SMTJRuntimeManager(
    instance_type=instance_type,
    instance_count=instance_count,
    execution_role=role,
)

runtime

In [ ]:
from amzn_nova_customization_sdk.model import NovaModelCustomizer
from amzn_nova_customization_sdk.model.model_enums import Model, TrainingMethod

job_name = "train-nova-2-lite-sft-peft"

customizer = NovaModelCustomizer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.SFT_LORA,
    infra=runtime,
    data_s3_path=train_dataset_s3_path,
    output_s3_path=f"{output_path_prefix}/{job_name}",
)

In [ ]:
overrides = {
    "max_epochs": 1,
    "lr": 2e-5,
    "warmup_steps": 50,
}

result = customizer.train(
    job_name=job_name,
    overrides=overrides,
)

In [ ]:
# Check job status
result.get_job_status()

In [ ]:
# Monitor training logs
customizer.get_logs()

You can monitor the job directly from your notebook output. You can also refer the SageMaker AI console, which shows the status of the job and the corresponding CloudWatch logs for governance and observability, as shown in the following screenshots.